# NWC 계수(γ) 진단 노트북 (v4)

**v4 변경점**: `from DATA import config` 방식으로 전환
- config.py가 내부에서 `from DATA.xxx import ...`를 사용하므로
  sys.path에는 **DATA의 부모 디렉토리**(stock_forecast)를 등록

## 실행 순서
1. Cell 1-A: `from DATA import config`
2. Cell 1-B: `get_db_info()` 호출 & key 매핑 & DB 접속 테스트
3. Cell 2-6: 진단 실행
4. Cell 7: 해석 가이드

## Cell 1-A · DATA 폴더 경로 자동 감지 & config import

In [1]:
import sys, os

# ── 프로젝트 루트 후보 경로 (DATA의 부모 디렉토리) ──────────
#    stock_forecast/DATA/config.py 가 내부에서
#    "from DATA.xxx import ..."을 사용하므로
#    sys.path에는 DATA가 아닌 그 부모(stock_forecast)를 넣어야 함
PROJECT_ROOT_CANDIDATES = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",  # ← 현재 노트북
    # 다른 머신(데스크톱 등)의 경로는 여기에 추가
]

project_root = None
for p in PROJECT_ROOT_CANDIDATES:
    # DATA 폴더가 존재하는지로 검증
    if os.path.isdir(os.path.join(p, "DATA")):
        project_root = p
        if p not in sys.path:
            sys.path.insert(0, p)
        print(f"[OK] 프로젝트 루트: {p}")
        break

if project_root is None:
    raise RuntimeError(
        "DATA 폴더를 포함한 프로젝트 루트를 찾지 못했습니다. "
        "PROJECT_ROOT_CANDIDATES에 stock_forecast 경로를 추가하세요."
    )

# ── DATA 패키지로 config 불러오기 ──────────────────────────
from DATA import config
print(f"[OK] config.py 로드됨: {config.__file__}")
print(f"[OK] get_db_info 존재 여부: {'get_db_info' in dir(config)}")


[OK] 프로젝트 루트: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast


C:\Users\Hoyoung_Park\PyCharmMiscProject\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] config.py 로드됨: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA\config.py
[OK] get_db_info 존재 여부: True


## Cell 1-B · `get_db_info()` 호출 & pymysql 호환 변환

`get_db_info()`가 어떤 key 이름으로 dict를 반환하는지 먼저 출력해서 확인합니다.

일반적인 key 이름 변형(host/Host/HOST, database/db/DB, password/pwd/passwd 등)을
자동으로 pymysql 표준 형태로 변환합니다. 만약 자동 변환이 실패하면 출력된 key 목록을
보고 Cell 1-B의 `KEY_ALIASES` 매핑만 수정하면 됩니다.

In [2]:
# ── 원본 dict 조회 ─────────────────────────────────────────
raw = config.get_db_info()
print("[get_db_info() 반환 dict]")
# password는 마스킹해서 출력
for k, v in raw.items():
    shown = "***" if any(s in k.lower() for s in ["pass", "pwd", "secret"]) else v
    print(f"  {k!r}: {shown!r}")

[get_db_info() 반환 dict]
  'host': 'hystox74.synology.me'
  'port': 3307
  'user': 'stox7412'
  'password': '***'
  'database': 'investar'


In [3]:
# ── pymysql 표준 key 이름으로 정규화 ────────────────────────
# pymysql.connect()가 받는 표준 인자: host, port, user, password, database, charset

KEY_ALIASES = {
    "host":     ["host", "HOST", "Host", "hostname", "server"],
    "port":     ["port", "PORT", "Port"],
    "user":     ["user", "USER", "User", "username", "uid"],
    "password": ["password", "PASSWORD", "Password", "passwd", "pwd", "PWD"],
    "database": ["database", "DATABASE", "Database", "db", "DB", "dbname", "schema"],
    "charset":  ["charset", "CHARSET", "encoding"],
}

DB_CONFIG = {}
for std_key, aliases in KEY_ALIASES.items():
    for alias in aliases:
        if alias in raw:
            DB_CONFIG[std_key] = raw[alias]
            break

# ── 필수 키 체크 & 기본값 ──────────────────────────────────
required = ["host", "user", "password", "database"]
missing  = [k for k in required if k not in DB_CONFIG]
if missing:
    print(f"[경고] 다음 키를 자동 매핑하지 못했습니다: {missing}")
    print(f"      raw 반환 key 목록: {list(raw.keys())}")
    print(f"      KEY_ALIASES에 본인 config의 실제 key 이름을 추가해주세요.")
    raise ValueError(f"필수 DB 접속 정보 누락: {missing}")

# port는 정수여야 함 (str인 경우 변환)
if "port" in DB_CONFIG:
    DB_CONFIG["port"] = int(DB_CONFIG["port"])
else:
    DB_CONFIG["port"] = 3307  # 메모리 기본값

# charset 기본값
DB_CONFIG.setdefault("charset", "utf8mb4")

# ── 최종 확인 (password 마스킹) ──────────────────────────────
print("[pymysql 호환 DB_CONFIG]")
for k, v in DB_CONFIG.items():
    shown = "***" if k == "password" else v
    print(f"  {k}: {shown}")

[pymysql 호환 DB_CONFIG]
  host: hystox74.synology.me
  port: 3307
  user: stox7412
  password: ***
  database: investar
  charset: utf8mb4


In [7]:
# ── 접속 테스트 & 라이브러리 로드 ───────────────────────────
import pandas as pd
import numpy as np
from scipy import stats
import pymysql
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 접속 테스트
try:
    _conn = pymysql.connect(**DB_CONFIG)
    _cur = _conn.cursor()
    _cur.execute("SELECT 1")
    _cur.fetchone()
    _conn.close()
    print("[OK] DB 접속 성공")
except Exception as e:
    print(f"[FAIL] DB 접속 실패: {e}")
    raise

# ── 진단 대상 ───────────────────────────────────────────────
TICKER    = "AUPH"
BS_TABLE  = "US_BS_from_FMP"   # ← 수정
IS_TABLE  = "US_IS_from_FMP"   # ← 수정

print(f"\n진단 대상: {TICKER}")
print(f"BS 테이블: {BS_TABLE}")
print(f"IS 테이블: {IS_TABLE}")

[OK] DB 접속 성공

진단 대상: AUPH
BS 테이블: US_BS_from_FMP
IS 테이블: US_IS_from_FMP


In [6]:
# # ── 실제 테이블 목록 조회 ───────────────────────────────────
# conn = pymysql.connect(**DB_CONFIG)
# cur = conn.cursor()
#
# cur.execute("SHOW TABLES")
# all_tables = [r[0] for r in cur.fetchall()]
# conn.close()
#
# print(f"investar DB 전체 테이블 ({len(all_tables)}개):\n")
# for t in all_tables:
#     print(f"  {t}")
#
# print("\n--- BS 후보 ---")
# for t in all_tables:
#     if any(k in t.lower() for k in ["balance", "bs"]):
#         print(f"  {t}")
#
# print("\n--- IS 후보 ---")
# for t in all_tables:
#     if any(k in t.lower() for k in ["income", "is"]):
#         print(f"  {t}")

investar DB 전체 테이블 (69개):

  Credit_Spread
  KSE_Price
  KTB_Yield_Daily
  KTB_daily
  Korea_Economy_Data
  Korea_company_valuation_ver2
  US_BS_from_FMP
  US_CF_from_FMP
  US_IS_from_FMP
  US_company_valuation_result
  US_funda
  US_fundm
  US_fundq
  bok_economic_indicators
  correlation_price_hscode
  correlation_table_DB
  firm_char
  fmp_financial_data
  hs_code_by_kr_monster_company
  korea_company_hscode_map
  korea_company_valuation_result
  korea_fcff_dcf_valuation
  korea_fs_data
  korea_fs_data_from_DART_V2
  korea_fs_data_from_DG
  korea_fs_data_roe_roa
  korea_monthly_trade_act_forecast_data
  korea_monthly_trade_data
  korea_monthly_trade_data_forecast
  korea_monthly_trade_data_v2
  korea_monthly_trade_forecast_v2
  korea_quarterly_trade_data
  korea_required_return_result
  korea_revenue_forecast_result
  korea_rim_valuation
  korea_valuation_quality_log
  ks_listed_company_daily_marketcap
  price_table
  psr_data
  sec_financial_data
  target_company_db
  target_hs_cod

In [13]:
# ══════════════════════════════════════════════════════════════════
# Long-format 구조 진단: AUPH의 item 종류 확인
# ══════════════════════════════════════════════════════════════════
conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor(pymysql.cursors.DictCursor)

# ── 1) BS의 item 종류 ─────────────────────────────────────
cur.execute(f"""
    SELECT DISTINCT item
    FROM {BS_TABLE}
    WHERE ticker = %s
    ORDER BY item
""", (TICKER,))
bs_items = [r["item"] for r in cur.fetchall()]
print(f"[BS] AUPH item 종류 {len(bs_items)}개:")
for i in bs_items:
    print(f"  {i}")

# ── 2) NWC 관련 item 자동 필터링 ──────────────────────────
nwc_keywords = ["current", "cash", "debt", "short", "asset", "liabilit", "receivable", "inventor", "payable"]
print(f"\n[BS] NWC 관련 item 후보:")
for i in bs_items:
    if any(k in i.lower() for k in nwc_keywords):
        print(f"  {i}")

# ── 3) IS의 item 종류 ─────────────────────────────────────
cur.execute(f"""
    SELECT DISTINCT item
    FROM {IS_TABLE}
    WHERE ticker = %s
    ORDER BY item
""", (TICKER,))
is_items = [r["item"] for r in cur.fetchall()]
print(f"\n[IS] AUPH item 종류 {len(is_items)}개:")
for i in is_items:
    print(f"  {i}")

# ── 4) 샘플 데이터 한 분기 ────────────────────────────────
cur.execute(f"""
    SELECT date, item, value, fs_name
    FROM {BS_TABLE}
    WHERE ticker = %s
    ORDER BY date DESC, item
    LIMIT 30
""", (TICKER,))
sample = pd.DataFrame(cur.fetchall())
print(f"\n[BS 최근 샘플]")
print(sample.to_string())

# ── 5) IS의 period 분포 확인 ──────────────────────────────
cur.execute(f"""
    SELECT period, COUNT(*) AS n
    FROM {IS_TABLE}
    WHERE ticker = %s
    GROUP BY period
""", (TICKER,))
print(f"\n[IS] period 분포:")
for r in cur.fetchall():
    print(f"  {r['period']}: {r['n']}개")

conn.close()

[BS] AUPH item 종류 18개:
  ao
  ap
  at
  be
  ca
  cash
  debt
  debtlt
  debtst
  intan
  invt
  ivao
  lo
  lt
  ppen
  pstk
  rec
  txp

[BS] NWC 관련 item 후보:
  cash
  debt
  debtlt
  debtst

[IS] AUPH item 종류 47개:
  cogs
  costAndExpenses
  costOfRevenue
  depreciationAndAmortization
  dp
  ebitda
  ebitdaratio
  eps
  epsdi
  epsdiluted
  generalAndAdministrativeExpenses
  gp
  grossProfit
  grossProfitRatio
  idit
  incomeBeforeTax
  incomeBeforeTaxRatio
  incomeTaxExpense
  interestExpense
  interestIncome
  netIncome
  netIncomeRatio
  ni
  nir
  operatingExpenses
  operatingIncome
  operatingIncomeRatio
  opir
  opiti
  otherExpenses
  pi
  pir
  researchAndDevelopmentExpenses
  revenue
  sale
  sellingAndMarketingExpenses
  sellingGeneralAndAdministrativeExpenses
  shrout
  shroutdi
  totalOtherIncomeExpensesNet
  txt
  weightedAverageShsOut
  weightedAverageShsOutDil
  xint
  xopr
  xrd
  xsga

[BS 최근 샘플]
         date    item          value fs_name
0  2026-01-31      ao 161,7

In [14]:
# ══════════════════════════════════════════════════════════════════
# AUPH BS의 실제 item별 값 확인 — NWC 계산 오류 근본 원인 규명
# ══════════════════════════════════════════════════════════════════
conn = pymysql.connect(**DB_CONFIG)
cur = conn.cursor(pymysql.cursors.DictCursor)

# ── 1) AUPH BS 전체 데이터 pivot ─────────────────────────────
cur.execute(f"""
    SELECT date, item, value
    FROM {BS_TABLE}
    WHERE ticker = %s
    ORDER BY date, item
""", (TICKER,))
bs_long = pd.DataFrame(cur.fetchall())
bs_long["value"] = pd.to_numeric(bs_long["value"], errors="coerce")

bs_wide = bs_long.pivot_table(index="date", columns="item", values="value", aggfunc="last")
print(f"[BS wide] shape = {bs_wide.shape}")
print(f"컬럼: {list(bs_wide.columns)}\n")

# ── 2) NWC 계산: 3가지 정의 비교 ────────────────────────────
# (a) Compustat 표준 풀 계산 (current liabilities = ap + debtst + txp 근사)
# (b) lt - debtlt 방식 (current liabilities = 전체부채 − 장기부채)
# (c) FCFF 코드가 "lt를 totalCurrentLiabilities로" 썼을 경우 (버그 재현)

bs_wide["nwc_correct_a"] = (
    (bs_wide.get("ca", 0) - bs_wide.get("cash", 0))
    - (bs_wide.get("ap", 0) + bs_wide.get("debtst", 0) + bs_wide.get("txp", 0))
)
bs_wide["nwc_correct_b"] = (
    (bs_wide.get("ca", 0) - bs_wide.get("cash", 0))
    - (bs_wide.get("lt", 0) - bs_wide.get("debtlt", 0) - bs_wide.get("debtst", 0))
)
bs_wide["nwc_buggy_c"] = (
    (bs_wide.get("ca", 0) - bs_wide.get("cash", 0))
    - (bs_wide.get("lt", 0) - bs_wide.get("debtst", 0))
)

# ── 3) IS에서 revenue 가져와서 ratio 비교 ────────────────────
cur.execute(f"""
    SELECT date, value AS revenue
    FROM {IS_TABLE}
    WHERE ticker = %s AND item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
    ORDER BY date
""", (TICKER,))
rev = pd.DataFrame(cur.fetchall())
rev["revenue"] = pd.to_numeric(rev["revenue"], errors="coerce")
rev = rev.set_index("date")

# ── 4) 통합 표 ───────────────────────────────────────────────
cmp = bs_wide[["ca","cash","ap","txp","debtst","debtlt","lt",
               "nwc_correct_a","nwc_correct_b","nwc_buggy_c"]].copy()
cmp = cmp.join(rev, how="left")
cmp = cmp.dropna(subset=["revenue"])
cmp["ratio_a"] = cmp["nwc_correct_a"] / cmp["revenue"]
cmp["ratio_b"] = cmp["nwc_correct_b"] / cmp["revenue"]
cmp["ratio_c_buggy"] = cmp["nwc_buggy_c"] / cmp["revenue"]

print("[AUPH BS wide + 3가지 NWC 정의 + ratio]")
print(cmp.to_string())

print("\n[ratio 중앙값 비교]")
print(f"  (a) 영업 current liab 근사:     median ratio = {cmp['ratio_a'].median():.3f}")
print(f"  (b) lt - debtlt (Balance 잔차): median ratio = {cmp['ratio_b'].median():.3f}")
print(f"  (c) lt 그대로 사용 (버그 추정): median ratio = {cmp['ratio_c_buggy'].median():.3f}")
print(f"\n[FCFF 모델이 출력한 γ = -125.16]")
print("  → (c)와 가까우면 버그 확정")

conn.close()

[BS wide] shape = (131, 18)
컬럼: ['ao', 'ap', 'at', 'be', 'ca', 'cash', 'debt', 'debtlt', 'debtst', 'intan', 'invt', 'ivao', 'lo', 'lt', 'ppen', 'pstk', 'rec', 'txp']

[AUPH BS wide + 3가지 NWC 정의 + ratio]
                       ca          cash           ap           txp       debtst  debtlt   lt  nwc_correct_a  nwc_correct_b    nwc_buggy_c       revenue  ratio_a  ratio_b  ratio_c_buggy
date                                                                                                                                                                                    
2024-09-30 440,459,000.00 37,142,000.00 1,007,000.00 71,906,000.00 8,878,000.00    0.00 0.00 321,526,000.00 412,195,000.00 412,195,000.00 67,771,000.00     4.74     6.08           6.08
2024-12-31 446,596,000.00 83,433,000.00         0.00 64,297,000.00 5,187,000.00    0.00 0.00 293,679,000.00 368,350,000.00 368,350,000.00 59,867,000.00     4.91     6.15           6.15
2025-03-31 405,762,000.00 66,428,000.00         0.00 62,3

## Cell 2 · BS / IS 원본 조회

In [10]:
conn = pymysql.connect(**DB_CONFIG)
cur  = conn.cursor(pymysql.cursors.DictCursor)

# ── BS 테이블 컬럼 확인 ─────────────────────────────────────
cur.execute(f"SHOW COLUMNS FROM {BS_TABLE}")
bs_cols = [r["Field"] for r in cur.fetchall()]
print(f"[{BS_TABLE}] 전체 컬럼 {len(bs_cols)}개")
nwc_related = [c for c in bs_cols
               if any(k in c.lower() for k in
                      ["current", "cash", "debt", "period", "date", "ticker", "accept"])]
print(f"NWC 관련 후보: {nwc_related}")

[US_BS_from_FMP] 전체 컬럼 7개
NWC 관련 후보: ['date', 'report_date', 'ticker', 'date_month']


In [11]:
# ── BS 조회 ────────────────────────────────────────────────
select_cols = ["date", "currentAssets", "cashAndCashEquivalents",
               "totalCurrentLiabilities", "shortTermDebt"]
optional = ["period", "acceptedDate", "calendarYear", "fillingDate"]
for c in optional:
    if c in bs_cols:
        select_cols.append(c)

cols_sql = ", ".join(f"`{c}`" for c in select_cols)
cur.execute(f"""
    SELECT {cols_sql}
    FROM {BS_TABLE}
    WHERE ticker = %s
    ORDER BY date
""", (TICKER,))
bs = pd.DataFrame(cur.fetchall())

# ── IS 조회 ────────────────────────────────────────────────
cur.execute(f"SHOW COLUMNS FROM {IS_TABLE}")
is_cols = [r["Field"] for r in cur.fetchall()]

is_select = ["date", "revenue"]
for c in optional:
    if c in is_cols:
        is_select.append(c)

is_cols_sql = ", ".join(f"`{c}`" for c in is_select)
cur.execute(f"""
    SELECT {is_cols_sql}
    FROM {IS_TABLE}
    WHERE ticker = %s
    ORDER BY date
""", (TICKER,))
inc = pd.DataFrame(cur.fetchall())

conn.close()

print(f"[BS] rows = {len(bs)}, cols = {list(bs.columns)}")
print(f"[IS] rows = {len(inc)}, cols = {list(inc.columns)}")

OperationalError: (1054, "Unknown column 'currentAssets' in 'field list'")

## Cell 3 · period 분포 & date 중복 확인 (**핵심**)

In [ ]:
print("="*60)
print("[BS] period 분포")
print("="*60)
if "period" in bs.columns:
    print(bs["period"].value_counts(dropna=False))
else:
    print("period 컬럼 없음 (분기/연간 구분 불가)")

print("\n" + "="*60)
print("[IS] period 분포")
print("="*60)
if "period" in inc.columns:
    print(inc["period"].value_counts(dropna=False))
else:
    print("period 컬럼 없음")

print("\n" + "="*60)
print("[BS] date 중복")
print("="*60)
dup_bs = bs[bs.duplicated("date", keep=False)].sort_values("date")
if dup_bs.empty:
    print("중복 없음")
else:
    print(f"중복 {len(dup_bs)}행:")
    print(dup_bs.to_string())

print("\n[IS] date 중복")
dup_is = inc[inc.duplicated("date", keep=False)].sort_values("date")
if dup_is.empty:
    print("중복 없음")
else:
    print(f"중복 {len(dup_is)}행:")
    print(dup_is.to_string())

## Cell 4 · NWC 계산 & merged 데이터 전체 확인

In [ ]:
# ── NWC 계산 (DCFModel과 동일) ───────────────────────────────
for c in ["currentAssets", "cashAndCashEquivalents",
          "totalCurrentLiabilities", "shortTermDebt"]:
    bs[c] = pd.to_numeric(bs[c], errors="coerce").fillna(0)

bs["nwc"] = ((bs["currentAssets"] - bs["cashAndCashEquivalents"])
             - (bs["totalCurrentLiabilities"] - bs["shortTermDebt"]))

inc["revenue"] = pd.to_numeric(inc["revenue"], errors="coerce")

bs_cols_to_merge = ["date", "nwc"]
if "period" in bs.columns:
    bs_cols_to_merge.append("period")

inc_cols_to_merge = ["date", "revenue"]
if "period" in inc.columns:
    inc_cols_to_merge.append("period")

merged = inc[inc_cols_to_merge].merge(
    bs[bs_cols_to_merge], on="date", how="inner",
    suffixes=("_is", "_bs")
).dropna(subset=["revenue", "nwc"])

merged = merged[merged["revenue"] > 0].sort_values("date").reset_index(drop=True)
merged["ratio"] = merged["nwc"] / merged["revenue"]

print(f"merged n = {len(merged)}")
print("\n[merged 전체]")
print(merged.to_string())

In [ ]:
# ── ratio 분포 & 이상치 ─────────────────────────────────────
print("="*60)
print("NWC / Revenue ratio 분포")
print("="*60)
print(merged["ratio"].describe())

med = merged["ratio"].median()
mad = (merged["ratio"] - med).abs().median()
threshold = 3 * mad if mad > 0 else 3 * merged["ratio"].std()

merged["is_outlier"] = (merged["ratio"] - med).abs() > threshold

print(f"\nmedian = {med:.4f}, MAD = {mad:.4f}, 임계값 = {threshold:.4f}")
print(f"\n[이상치 행 ({merged['is_outlier'].sum()}개)]")
outliers = merged[merged["is_outlier"]]
if outliers.empty:
    print("없음")
else:
    print(outliers.to_string())

## Cell 5 · OLS γ vs median ratio 비교

In [ ]:
x = merged["revenue"].values
y = merged["nwc"].values

# 1) OLS (절편 포함) — 현재 DCFModel 방식
slope, intercept, r, p, se = stats.linregress(x, y)
r2 = r ** 2

# 2) 무절편 OLS
slope_no_intercept = (x * y).sum() / (x * x).sum()

# 3) median ratio
gamma_median = merged["ratio"].median()

# 4) winsorized median
ratios = merged["ratio"].copy()
lo, hi = ratios.quantile(0.05), ratios.quantile(0.95)
gamma_median_w = ratios.clip(lo, hi).median()

# 5) outlier 제거 후 OLS
clean = merged[~merged["is_outlier"]]
if len(clean) >= 10:
    slope_clean, intercept_clean, r_clean, _, _ = stats.linregress(
        clean["revenue"], clean["nwc"])
    r2_clean = r_clean ** 2
else:
    slope_clean = intercept_clean = r2_clean = np.nan

print("="*75)
print("γ 추정 방식별 비교")
print("="*75)
print(f"{'방식':<35} {'γ':>15} {'비고':<20}")
print("-"*75)
print(f"{'OLS (절편 포함, 현재 채택)':<35} {slope:>15,.4f}  R²={r2:.3f}, intercept={intercept:,.0f}")
print(f"{'OLS (무절편)':<35} {slope_no_intercept:>15,.4f}")
print(f"{'median ratio':<35} {gamma_median:>15,.4f}")
print(f"{'median ratio (5-95% winsorized)':<35} {gamma_median_w:>15,.4f}")
if not np.isnan(slope_clean):
    print(f"{'OLS (outlier 제거)':<35} {slope_clean:>15,.4f}  R²={r2_clean:.3f}")
else:
    print(f"{'OLS (outlier 제거)':<35} {'N/A':>15}")
print("="*75)

# 예측 영향 — AUPH 2026Q1 sales=60.4M 기준
sales_example = 60_424_550
print(f"\n예상 NWC @ 2026Q1 sales=${sales_example:,.0f}")
print(f"  OLS 절편포함:  ${slope * sales_example:>18,.0f}")
print(f"  OLS 무절편:    ${slope_no_intercept * sales_example:>18,.0f}")
print(f"  median:        ${gamma_median * sales_example:>18,.0f}")
print(f"  winsor median: ${gamma_median_w * sales_example:>18,.0f}")
if not np.isnan(slope_clean):
    print(f"  clean OLS:     ${slope_clean * sales_example:>18,.0f}")

## Cell 6 · 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Revenue vs NWC 산점도 + 4개 회귀선
ax = axes[0]
colors = np.where(merged["is_outlier"], "red", "steelblue")
ax.scatter(merged["revenue"], merged["nwc"], c=colors, s=60, alpha=0.7,
           edgecolors="black", linewidths=0.5)

x_range = np.linspace(0, merged["revenue"].max() * 1.1, 100)
ax.plot(x_range, slope * x_range + intercept, "r-",
        label=f"OLS w/ intercept (gamma={slope:.2f})", linewidth=2)
ax.plot(x_range, slope_no_intercept * x_range, "g--",
        label=f"OLS no-intercept (gamma={slope_no_intercept:.2f})", linewidth=2)
ax.plot(x_range, gamma_median * x_range, "b:",
        label=f"median ratio (gamma={gamma_median:.2f})", linewidth=2)
if not np.isnan(slope_clean):
    ax.plot(x_range, slope_clean * x_range + intercept_clean, "purple", linestyle="-.",
            label=f"OLS clean (gamma={slope_clean:.2f})", linewidth=2)

for _, row in merged[merged["is_outlier"]].iterrows():
    ax.annotate(str(row["date"])[:10], (row["revenue"], row["nwc"]),
                fontsize=8, color="red", xytext=(5, 5), textcoords="offset points")

ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Revenue")
ax.set_ylabel("NWC")
ax.set_title(f"{TICKER}: Revenue vs NWC (red = outliers)")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)

# 오른쪽: ratio 시계열
ax = axes[1]
merged_plot = merged.copy()
merged_plot["date"] = pd.to_datetime(merged_plot["date"])
ax.plot(merged_plot["date"], merged_plot["ratio"], "o-", color="steelblue")
ax.scatter(merged_plot.loc[merged_plot["is_outlier"], "date"],
           merged_plot.loc[merged_plot["is_outlier"], "ratio"],
           color="red", s=100, zorder=5, label="outlier")
ax.axhline(gamma_median, color="blue", linestyle=":",
           label=f"median = {gamma_median:.2f}")
ax.axhline(slope, color="red", linestyle="--",
           label=f"OLS gamma = {slope:.2f}")
ax.set_xlabel("Date")
ax.set_ylabel("NWC / Revenue")
ax.set_title(f"{TICKER}: NWC/Revenue ratio time series")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Cell 7 · 해석 가이드

### 결과 해석 체크리스트

**Q1. Cell 3의 period 분포에서 'FY' / 'annual' / 'TTM' 행이 BS에 섞여 있나?**
- YES → 바로 이것이 주범. `estimate_nwc_coef`에서 merge 전 분기 필터링 필요.
- NO → Q2로

**Q2. date 중복 행이 있나?**
- YES → FMP 다중 보고 이슈. `acceptedDate` 최신 기준 drop_duplicates 필요.
- NO → Q3으로

**Q3. Cell 5에서 OLS slope vs median ratio 차이가 얼마나 나나?**
- 10배 이상 → OLS가 소수 outlier에 leverage됨. median-first 전략 전환 권장.
- 2-3배 이내 → 다른 버그 의심.

**Q4. Cell 6 왼쪽 그래프에서 빨간 점(outlier)이 회귀선을 크게 끌고 있나?**
- YES → leverage point 확정. clean OLS 또는 robust regression 필요.

In [15]:
# ══════════════════════════════════════════════════════════════════
# FMP API 직접 호출 → AUPH NWC γ 재현 진단 (v4 노트북과 별개, 독립 실행 가능)
# ══════════════════════════════════════════════════════════════════
import requests, time
import pandas as pd
import numpy as np
from scipy import stats

FMP_API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"   # 메모리 기록값 기준
FMP_BASE    = "https://financialmodelingprep.com/api/v3"
TICKER      = "AUPH"

def fmp_get(endpoint, ticker, period="quarter", limit=120):
    url = f"{FMP_BASE}/{endpoint}/{ticker}"
    r = requests.get(url, params={"period": period, "limit": limit,
                                   "apikey": FMP_API_KEY}, timeout=20)
    r.raise_for_status()
    data = r.json()
    return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame()

# ── 1) FMP에서 BS / IS 원본 조회 ───────────────────────────
bs_api = fmp_get("balance-sheet-statement", TICKER)
time.sleep(0.3)
is_api = fmp_get("income-statement", TICKER)

print(f"[BS API] rows={len(bs_api)}, cols={len(bs_api.columns)}")
print(f"[IS API] rows={len(is_api)}")

# ── 2) NWC 4대 항목 존재 확인 ──────────────────────────────
need_cols = ["currentAssets", "cashAndCashEquivalents",
             "totalCurrentLiabilities", "shortTermDebt"]
print("\n[BS 필수 컬럼 존재 여부]")
for c in need_cols:
    has = c in bs_api.columns
    mark = "✓" if has else "❌"
    print(f"  {mark} {c}")

for c in need_cols:
    if c not in bs_api.columns:
        bs_api[c] = 0.0
    bs_api[c] = pd.to_numeric(bs_api[c], errors="coerce").fillna(0)

is_api["revenue"] = pd.to_numeric(is_api["revenue"], errors="coerce")

# ── 3) NWC 계산 + merge (FCFF 모델 로직과 동일) ────────────
bs_api["nwc"] = (
    (bs_api["currentAssets"] - bs_api["cashAndCashEquivalents"])
    - (bs_api["totalCurrentLiabilities"] - bs_api["shortTermDebt"])
)

bs_api["date"] = pd.to_datetime(bs_api["date"])
is_api["date"] = pd.to_datetime(is_api["date"])

merged = is_api[["date","revenue"]].merge(
    bs_api[["date","nwc","currentAssets","cashAndCashEquivalents",
            "totalCurrentLiabilities","shortTermDebt","period"]],
    on="date", how="inner"
).dropna()
merged = merged[merged["revenue"] > 0].sort_values("date").reset_index(drop=True)
merged["ratio"] = merged["nwc"] / merged["revenue"]

# ── 4) 출력 ────────────────────────────────────────────────
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
print("\n[전체 분기별 BS 구성요소 + NWC + ratio]")
print(merged[["date","period","revenue","currentAssets","cashAndCashEquivalents",
              "totalCurrentLiabilities","shortTermDebt","nwc","ratio"]].to_string())

# ── 5) γ 추정 재현 ──────────────────────────────────────────
print("\n" + "="*60)
print(f"ratio 통계: n={len(merged)}")
print(f"  median = {merged['ratio'].median():.4f}")
print(f"  mean   = {merged['ratio'].mean():.4f}")
print(f"  min    = {merged['ratio'].min():.4f}")
print(f"  max    = {merged['ratio'].max():.4f}")

slope, intercept, r, _, _ = stats.linregress(merged["revenue"], merged["nwc"])
print(f"\nOLS (절편 포함): slope(γ) = {slope:.4f}, R² = {r**2:.4f}, "
      f"intercept = {intercept:,.0f}")
print(f"→ FCFF 출력의 γ = -125.16과 비교해주세요")

# ── 6) period 분포 (Q4 annual 혼입 체크) ────────────────────
print(f"\n[period 분포]")
print(merged["period"].value_counts())

[BS API] rows=107, cols=54
[IS API] rows=107

[BS 필수 컬럼 존재 여부]
  ❌ currentAssets
  ✓ cashAndCashEquivalents
  ✓ totalCurrentLiabilities
  ✓ shortTermDebt

[전체 분기별 BS 구성요소 + NWC + ratio]
          date period    revenue  currentAssets  cashAndCashEquivalents  totalCurrentLiabilities  shortTermDebt          nwc   ratio
0   1999-02-28     Q1    397,746              0                 1193238                  1657275        464,037   -2,386,476      -6
1   1999-05-31     Q2    271,280              0                       0                  1763321        949,480     -813,841      -3
2   1999-08-31     Q3    200,729              0                  133819                   802917         66,909     -869,827      -4
3   1999-11-30     Q4    488,370              0                 2580575                   882828         67,909   -3,395,494      -7
4   2000-02-29     Q1    275,660              0                 2894435                   620236         68,915   -3,445,756     -13
5   2000-05-31  